# 08 Hybrid Retrieval (Document-Unknown)

## Σκοπός
Σε αυτό το notebook:

- φορτώνεται το FinanceBench working dataset
- φορτώνονται chunks και embeddings
- χρησιμοποιείται document-unknown retrieval
- εκτελείται dense retrieval μέσα στο σωστό document
- εκτελείται BM25 retrieval μέσα στο σωστό document
- συνδυάζονται dense και BM25 με Reciprocal Rank Fusion (RRF)
- αποθηκεύονται τα hybrid retrieval results

Στόχος είναι η δημιουργία ενός baseline hybrid retriever για το document-known setting.

In [ ]:
import json
import math
import re
from collections import Counter
from pathlib import Path
import warnings
import torch

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [ ]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 180)

IN_KAGGLE = Path("/kaggle/working").exists()
print("IN_KAGGLE:", IN_KAGGLE)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
EMBEDDING_MODEL = "BAAI/bge-m3"
# Persist 20 fused candidates so notebook 09 can rerank the declared pool.
TOP_K = 20
EVALUATION_TOP_K = 10

DENSE_CANDIDATES_K = 20
BM25_CANDIDATES_K = 10
RRF_K = 30

USE_QUERY_LIMIT = False
QUERY_LIMIT = 50

retrieval_config = {
    "retrieval_type": "hybrid",
    "embedding_model": EMBEDDING_MODEL,
    "top_k": TOP_K,
    "evaluation_top_k": EVALUATION_TOP_K,
    "dense_candidates_k": DENSE_CANDIDATES_K,
    "bm25_candidates_k": BM25_CANDIDATES_K,
    "rrf_k": RRF_K,
    "document_known": False,
    "query_expansion": True
}

retrieval_config

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

CHUNKS_DIR = PROCESSED_DIR / "chunks"
EMBEDDINGS_DIR = PROCESSED_DIR / "embeddings"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"

WORKING_DATASET_CSV_PATH = INTERIM_DIR / "financebench_open_source_working.csv"
WORKING_DATASET_PARQUET_PATH = INTERIM_DIR / "financebench_open_source_working.parquet"

CHUNKS_CSV_PATH = CHUNKS_DIR / "financebench_chunks.csv"
CHUNKS_PARQUET_PATH = CHUNKS_DIR / "financebench_chunks.parquet"

EMBEDDINGS_MATRIX_PATH = EMBEDDINGS_DIR / "chunk_embeddings.npy"
EMBEDDINGS_METADATA_CSV_PATH = EMBEDDINGS_DIR / "chunk_embeddings_metadata.csv"
EMBEDDINGS_METADATA_PARQUET_PATH = EMBEDDINGS_DIR / "chunk_embeddings_metadata.parquet"

RETRIEVAL_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.csv"
RETRIEVAL_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.parquet"
RETRIEVAL_MANIFEST_PATH = RETRIEVAL_DIR / "retrieval_manifest_hybrid.csv"
RETRIEVAL_STATS_PATH = RETRIEVAL_DIR / "retrieval_stats_hybrid.json"

RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("CHUNKS_PARQUET_PATH:", CHUNKS_PARQUET_PATH)
print("RETRIEVAL_DIR:", RETRIEVAL_DIR)

In [ ]:
if WORKING_DATASET_PARQUET_PATH.exists():
    query_df = pd.read_parquet(WORKING_DATASET_PARQUET_PATH)
elif WORKING_DATASET_CSV_PATH.exists():
    query_df = pd.read_csv(WORKING_DATASET_CSV_PATH)
else:
    raise FileNotFoundError("Working dataset not found.")

print("query_df shape before filtering:", query_df.shape)
query_df.head(2)

In [ ]:
required_cols = ["question", "financebench_id"]

missing_cols = [c for c in required_cols if c not in query_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in query_df: {missing_cols}")

query_df = query_df.copy()

if "question_clean" not in query_df.columns:
    query_df["question_clean"] = query_df["question"].astype(str)

if "doc_name" not in query_df.columns:
    query_df["doc_name"] = None

query_df["question_clean"] = (
    query_df["question_clean"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

query_df = query_df[query_df["question_clean"].notna()].copy().reset_index(drop=True)

if USE_QUERY_LIMIT:
    query_df = query_df.head(QUERY_LIMIT).copy().reset_index(drop=True)

print("query_df shape after filtering:", query_df.shape)
print(query_df.columns.tolist())
query_df[["financebench_id", "question", "question_clean", "doc_name"]].head(3)

In [ ]:
if CHUNKS_PARQUET_PATH.exists():
    chunks_df = pd.read_parquet(CHUNKS_PARQUET_PATH)
elif CHUNKS_CSV_PATH.exists():
    chunks_df = pd.read_csv(CHUNKS_CSV_PATH)
else:
    raise FileNotFoundError("Chunks file not found.")

print("chunks_df shape:", chunks_df.shape)
print(chunks_df.columns.tolist())
chunks_df.head(2)

In [ ]:
if EMBEDDINGS_METADATA_PARQUET_PATH.exists():
    embeddings_metadata_df = pd.read_parquet(EMBEDDINGS_METADATA_PARQUET_PATH)
elif EMBEDDINGS_METADATA_CSV_PATH.exists():
    embeddings_metadata_df = pd.read_csv(EMBEDDINGS_METADATA_CSV_PATH)
else:
    raise FileNotFoundError("Embeddings metadata file not found.")

if not EMBEDDINGS_MATRIX_PATH.exists():
    raise FileNotFoundError("Embeddings matrix .npy file not found.")

embeddings_matrix = np.load(EMBEDDINGS_MATRIX_PATH)

print("embeddings_metadata_df shape:", embeddings_metadata_df.shape)
print("embeddings_matrix shape:", embeddings_matrix.shape)

assert len(embeddings_metadata_df) == len(embeddings_matrix), "Metadata and embeddings row count mismatch"

print(embeddings_metadata_df.columns.tolist())
embeddings_metadata_df.head(2)

In [ ]:
retrieval_corpus_df = chunks_df.copy().reset_index(drop=True)

assert len(retrieval_corpus_df) == len(embeddings_matrix), \
    "Chunks dataframe and embeddings matrix row count mismatch"

required_chunk_cols = ["chunk_id", "doc_id", "chunk_text"]

missing_chunk_cols = [c for c in required_chunk_cols if c not in retrieval_corpus_df.columns]
if missing_chunk_cols:
    raise ValueError(f"Missing required columns in retrieval_corpus_df: {missing_chunk_cols}")

print("retrieval_corpus_df shape:", retrieval_corpus_df.shape)
print(retrieval_corpus_df.columns.tolist())
retrieval_corpus_df.head(2)

In [ ]:
FINANCE_ALIAS_MAP = {
    "ppe": [
        "property plant equipment",
        "property plant and equipment",
        "property plant and equipment net",
        "pp&e",
        "balance sheet"
    ],
    "net ppe": [
        "property plant and equipment net",
        "net property plant equipment",
        "property plant equipment net",
        "balance sheet"
    ],
    "capex": [
        "capital expenditure",
        "capital expenditures",
        "purchases of property plant and equipment",
        "purchases of property, plant and equipment",
        "cash flow statement"
    ],
    "cogs": [
        "cost of goods sold",
        "cost of sales",
        "cost of products sold",
        "income statement"
    ],
    "ebitda": [
        "earnings before interest taxes depreciation and amortization",
        "non-gaap operating performance"
    ],
    "opex": [
        "operating expenses",
        "selling general and administrative",
        "sg&a"
    ],
    "gross margin": [
        "gross profit margin",
        "gross profit",
        "income statement"
    ],
    "operating cash flow": [
        "net cash provided by operating activities",
        "cash flow statement"
    ],
    "free cash flow": [
        "free cash flow",
        "net cash provided by operating activities",
        "capital expenditures"
    ],
    "working capital": [
        "current assets",
        "current liabilities",
        "balance sheet"
    ],
    "payout ratio": [
        "dividends",
        "net income",
        "dividend payout ratio"
    ],
    "retention ratio": [
        "retained earnings ratio",
        "dividend payout ratio",
        "dividends",
        "net income"
    ],
}

In [ ]:
def normalize_query_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def expand_finance_query(query: str) -> str:
    q = normalize_query_text(query)
    expansions = []

    ordered_aliases = sorted(FINANCE_ALIAS_MAP.keys(), key=len, reverse=True)

    for alias in ordered_aliases:
        if alias in q:
            expansions.extend(FINANCE_ALIAS_MAP[alias])

    if "balance sheet" in q:
        expansions.extend(["balance sheet", "assets liabilities equity"])
    if "cash flow" in q:
        expansions.extend(["cash flow statement", "operating activities investing activities financing activities"])
    if "income statement" in q:
        expansions.extend(["income statement", "net sales operating income net income"])
    if "year end" in q or "year-end" in q:
        expansions.extend(["at december 31", "balance sheet"])

    seen = set()
    cleaned_expansions = []

    for item in expansions:
        item = item.strip().lower()
        if item and item not in seen:
            seen.add(item)
            cleaned_expansions.append(item)

    if cleaned_expansions:
        return q + " " + " ".join(cleaned_expansions)

    return q

In [ ]:
query_df["expanded_question"] = query_df["question_clean"].apply(expand_finance_query)

query_df[["financebench_id", "question_clean", "expanded_question"]].head(10)

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("Loaded embedding model:", EMBEDDING_MODEL)

In [ ]:
def embed_queries(texts):
    vectors = embedding_model.encode(
        texts,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.asarray(vectors, dtype="float32")

In [ ]:
def build_global_index(embeddings: np.ndarray):
    vectors = embeddings.astype("float32").copy()
    faiss.normalize_L2(vectors)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def search_dense_global(query_vector: np.ndarray, index, top_k: int):
    scores, indices = index.search(query_vector, k=min(top_k, index.ntotal))
    return scores[0], indices[0]

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-zA-Z0-9&\.]+")


def tokenize_for_bm25(text: str):
    text = str(text).lower()
    return TOKEN_PATTERN.findall(text)


bm25_docs_tokens = [tokenize_for_bm25(text) for text in retrieval_corpus_df["chunk_text"].tolist()]
doc_freq = Counter()
doc_lens = []

for tokens in bm25_docs_tokens:
    doc_lens.append(len(tokens))
    for tok in set(tokens):
        doc_freq[tok] += 1

N_DOCS = len(bm25_docs_tokens)
AVG_DOC_LEN = sum(doc_lens) / max(N_DOCS, 1)

BM25_K1 = 1.5
BM25_B = 0.75

print("N_DOCS:", N_DOCS)
print("AVG_DOC_LEN:", AVG_DOC_LEN)

In [ ]:
def bm25_score_query(query_text: str, top_k: int = BM25_CANDIDATES_K):
    query_tokens = tokenize_for_bm25(query_text)
    if not query_tokens:
        return []

    scores = np.zeros(N_DOCS, dtype=np.float32)

    query_token_counts = Counter(query_tokens)

    for token, qtf in query_token_counts.items():
        df = doc_freq.get(token, 0)
        if df == 0:
            continue

        idf = math.log(1 + (N_DOCS - df + 0.5) / (df + 0.5))

        for doc_idx, doc_tokens in enumerate(bm25_docs_tokens):
            tf = doc_tokens.count(token)
            if tf == 0:
                continue

            doc_len = doc_lens[doc_idx]
            denom = tf + BM25_K1 * (1 - BM25_B + BM25_B * (doc_len / AVG_DOC_LEN))
            score = idf * ((tf * (BM25_K1 + 1)) / denom)
            scores[doc_idx] += score

    if np.all(scores == 0):
        return []

    top_indices = np.argsort(-scores)[:top_k]
    results = [(int(idx), float(scores[idx])) for idx in top_indices if scores[idx] > 0]
    return results

In [ ]:
DENSE_WEIGHT = 1.0
BM25_WEIGHT = 0.2

def reciprocal_rank_fusion(dense_results, bm25_results, rrf_k=RRF_K):
    fused = {}

    for rank, (global_idx, score) in enumerate(dense_results, start=1):
        if global_idx not in fused:
            fused[global_idx] = {
                "global_idx": global_idx,
                "dense_rank": None,
                "dense_score": None,
                "bm25_rank": None,
                "bm25_score": None,
                "rrf_score": 0.0,
            }
        fused[global_idx]["dense_rank"] = rank
        fused[global_idx]["dense_score"] = float(score)
        fused[global_idx]["rrf_score"] += DENSE_WEIGHT / (rrf_k + rank)

    for rank, (global_idx, score) in enumerate(bm25_results, start=1):
        if global_idx not in fused:
            fused[global_idx] = {
                "global_idx": global_idx,
                "dense_rank": None,
                "dense_score": None,
                "bm25_rank": None,
                "bm25_score": None,
                "rrf_score": 0.0,
            }
        fused[global_idx]["bm25_rank"] = rank
        fused[global_idx]["bm25_score"] = float(score)
        fused[global_idx]["rrf_score"] += BM25_WEIGHT / (rrf_k + rank)

    fused_list = list(fused.values())
    fused_list = sorted(fused_list, key=lambda x: x["rrf_score"], reverse=True)
    return fused_list[:TOP_K]

In [ ]:
global_index = build_global_index(embeddings_matrix)
print("Global dense index size:", global_index.ntotal)

In [ ]:
query_vectors = embed_queries(query_df["expanded_question"].tolist())
np.save(EMBEDDINGS_DIR / "query_embeddings_grid_search.npy", query_vectors)
print("query_vectors shape:", query_vectors.shape)

In [ ]:
retrieval_records = []
manifest_records = []

for query_idx, (_, row) in enumerate(
    tqdm(query_df.iterrows(), total=len(query_df), desc="Running hybrid retrieval (document-unknown)")
):
    financebench_id = row.get("financebench_id")
    expected_doc_name = row.get("doc_name")
    question = row["question"]
    question_clean = row["question_clean"]
    expanded_question = row["expanded_question"]

    manifest_record = {
        "query_row": query_idx,
        "financebench_id": financebench_id,
        "question": question,
        "expected_doc_name": expected_doc_name,
        "status": None,
        "error_message": None,
        "n_results": 0
    }

    try:
        qvec = query_vectors[query_idx].reshape(1, -1)

        dense_scores, dense_indices = search_dense_global(
            query_vector=qvec,
            index=global_index,
            top_k=DENSE_CANDIDATES_K
        )
        dense_results = [(int(idx), float(score)) for idx, score in zip(dense_indices, dense_scores)]

        bm25_results = bm25_score_query(
            query_text=expanded_question,
            top_k=BM25_CANDIDATES_K
        )

        fused_results = reciprocal_rank_fusion(
            dense_results=dense_results,
            bm25_results=bm25_results,
            rrf_k=RRF_K
        )

        for rank, item in enumerate(fused_results, start=1):
            global_idx = int(item["global_idx"])
            matched_row = retrieval_corpus_df.iloc[global_idx]

            retrieval_records.append({
                "query_row": query_idx,
                "financebench_id": financebench_id,
                "question": question,
                "question_clean": question_clean,
                "expanded_question": expanded_question,
                "expected_doc_name": expected_doc_name,
                "expected_company": row.get("company"),
                "retrieved_rank": rank,
                "rrf_score": float(item["rrf_score"]),
                "dense_rank": item["dense_rank"],
                "dense_score": item["dense_score"],
                "bm25_rank": item["bm25_rank"],
                "bm25_score": item["bm25_score"],
                "embedding_row_idx": global_idx,
                "chunk_id": matched_row["chunk_id"],
                "retrieved_doc_id": matched_row["doc_id"],
                "chunk_index": matched_row.get("chunk_index"),
                "chunk_text": matched_row["chunk_text"],
                "char_count": matched_row.get("char_count"),
                "token_estimate": matched_row.get("token_estimate"),
            })

        manifest_record["status"] = "success"
        manifest_record["n_results"] = len(fused_results)

    except Exception as e:
        manifest_record["status"] = "error"
        manifest_record["error_message"] = str(e)

    manifest_records.append(manifest_record)

retrieval_results_df = pd.DataFrame(retrieval_records)
retrieval_manifest_df = pd.DataFrame(manifest_records)

print("retrieval_results_df shape:", retrieval_results_df.shape)
print("retrieval_manifest_df shape:", retrieval_manifest_df.shape)

In [ ]:
if "expected_doc_name" not in retrieval_results_df.columns:
    retrieval_results_df["expected_doc_name"] = None

retrieval_results_df["doc_match"] = (
    retrieval_results_df["expected_doc_name"].fillna("").astype(str)
    == retrieval_results_df["retrieved_doc_id"].fillna("").astype(str)
)

retrieval_results_df[[
    "financebench_id",
    "retrieved_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "rrf_score"
]].head(15)

In [ ]:
print("retrieval_results_df shape:", retrieval_results_df.shape)
print("retrieval_results_df columns:", retrieval_results_df.columns.tolist())

print("retrieval_manifest_df shape:", retrieval_manifest_df.shape)
print(retrieval_manifest_df["status"].value_counts(dropna=False))

retrieval_manifest_df[["financebench_id", "status", "error_message"]].head(20)

In [ ]:
top1_df = retrieval_results_df[retrieval_results_df["retrieved_rank"] == 1].copy()

top1_doc_match_rate = float(top1_df["doc_match"].mean()) if len(top1_df) else 0.0
evaluation_results_df = retrieval_results_df[
    retrieval_results_df["retrieved_rank"] <= EVALUATION_TOP_K
]
topk_doc_match_rate = float(
    evaluation_results_df.groupby("financebench_id")["doc_match"].max().mean()
) if len(evaluation_results_df) else 0.0

summary_df = pd.DataFrame([{
    "n_queries": int(query_df["financebench_id"].nunique()),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{EVALUATION_TOP_K}_doc_match_rate": topk_doc_match_rate
}])

summary_df

In [ ]:
retrieval_results_df.to_csv(RETRIEVAL_RESULTS_CSV_PATH, index=False, encoding="utf-8")
retrieval_results_df.to_parquet(RETRIEVAL_RESULTS_PARQUET_PATH, index=False)

retrieval_manifest_df.to_csv(RETRIEVAL_MANIFEST_PATH, index=False, encoding="utf-8")

print("Αποθηκεύτηκαν:")
print("-", RETRIEVAL_RESULTS_CSV_PATH)
print("-", RETRIEVAL_RESULTS_PARQUET_PATH)
print("-", RETRIEVAL_MANIFEST_PATH)


In [ ]:
retrieval_stats = {
    "retrieval_type": "hybrid",
    "document_known": False,
    "query_expansion": True,
    "embedding_model": EMBEDDING_MODEL,
    "top_k": TOP_K,
    "evaluation_top_k": EVALUATION_TOP_K,
    "dense_candidates_k": DENSE_CANDIDATES_K,
    "bm25_candidates_k": BM25_CANDIDATES_K,
    "rrf_k": RRF_K,
    "n_queries": int(query_df["financebench_id"].nunique()),
    "n_result_rows": int(len(retrieval_results_df)),
    "n_manifest_rows": int(len(retrieval_manifest_df)),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{EVALUATION_TOP_K}_doc_match_rate": topk_doc_match_rate,
    "results_csv": str(RETRIEVAL_RESULTS_CSV_PATH),
    "results_parquet": str(RETRIEVAL_RESULTS_PARQUET_PATH),
    "manifest_csv": str(RETRIEVAL_MANIFEST_PATH),
}

with open(RETRIEVAL_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(retrieval_stats, f, indent=2, ensure_ascii=False)

print("Saved stats:", RETRIEVAL_STATS_PATH)
retrieval_stats

In [ ]:
retrieval_results_df[[
    "financebench_id",
    "question",
    "expanded_question",
    "retrieved_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "rrf_score",
    "chunk_id"
]].head(20)

## Συμπέρασμα

Σε αυτό το notebook:

- εκτελέστηκε hybrid retrieval
- χρησιμοποιήθηκαν document-unknown retrieval
- εφαρμόστηκε finance-aware query expansion
- συνδυάστηκε dense retrieval και BM25
- ενώθηκαν τα αποτελέσματα με Reciprocal Rank Fusion (RRF)

